In [10]:
# dependencies
import pandas as pd
from pathlib import Path
import re
import os

In [11]:
# root directory
ROOT = Path.cwd()
FILE = ROOT / 'data/tagged_posts.csv'

In [12]:
# load the file 
df = pd.read_csv(FILE)

# exract the rows whoes tag is missing 
tb_tag = df[df['tags'].isna()].reset_index()

# record the index of the rows with missing tags
idx = tb_tag.index.tolist()

In [13]:
# rename the column 
tb_tag = tb_tag.rename(columns={'index': 'original_index'})

# store the result to excel file for manual labelling
col_keep = ['original_index', 'title', 'link', 'cleaned_text', 'tags']
tb_tag.to_excel(ROOT/ 'data/tb_tagged.xlsx', columns=col_keep, index=False)
print('File created!')

File created!


In [22]:
# load the manully labelled data 
labelled = pd.read_excel(ROOT/'data/tb_tagged.xlsx')

In [23]:
# merge the first tag and second tag into a list
# drop missing value when converting to a list
labelled["tags"] = [
    [t1, t2] if pd.notna(t2) else [t1] for t1, t2 in zip(labelled['tag1'], labelled['tag2'])
]
labelled['tags']

0                            [Programme insights]
1                            [Programme insights]
2                                  [Student life]
3                    [Life hacks, Life in Sweden]
4                                  [Student life]
                          ...                    
160                                  [Life hacks]
161                                  [Life hacks]
162                                    [About me]
163    [Programme insights, Student associations]
164             [Swedish culture, Life in Sweden]
Name: tags, Length: 165, dtype: object

In [16]:
# make sure the index align with the orignal df
labelled_aligned = labelled.set_index('original_index')

In [17]:
# merged the labelled data back to the original df
df.loc[labelled_aligned.index, 'tags'] = labelled_aligned['tags'].astype(str).values

In [18]:
# make sure the data is complete
df.isna().sum()

title           0
link            0
cleaned_text    0
Topic           0
tags            0
dtype: int64

In [21]:
# save the result to csv file
file_path = ROOT / 'data/cmp_tagged_posts.csv'
if os.path.isfile(file_path):
    print('File already exist')
else:
    df.to_csv(file_path, mode='w', index=False)
    print('File saved!')

File saved!
